# Poisson Regression with a Regularized Horseshoe Prior: Beta-Bayes Calibration

Extends the beta-Bayes calibration framework (well-specified threshold tau*, observed
discrepancy d_obs, root-found beta*) to a non-conjugate, non-Gaussian model: Poisson
regression with a sparsity-inducing regularized horseshoe prior (Piironen & Vehtari),
fit via NUTS (NumPyro/JAX). All reusable model/DGP/inference code lives in
`poisson_horseshoe_utils.py`; this notebook runs the actual calibration experiment.

Setup: p=10 covariates, 3 truly nonzero (b_true = [1.0, -0.75, 0.5, 0,...,0], b0=0.5),
n=500 observations per dataset, R=30 independent replicate pairs per Monte Carlo estimate.

In [1]:
import functools
import numpy as np

import poisson_horseshoe_utils as phu

p, n = 10, 500
b0_true = 0.5
b_true = np.array([1.0, -0.75, 0.5, 0, 0, 0, 0, 0, 0, 0])  # 3 nonzero, 7 null
R_NB = 5.0  # Negative-Binomial overdispersion (misspecified DGP)
tau0 = phu.tau0_default(p, p0=3, n=n)

R = 30  # replicate pairs per Monte Carlo estimate
NUM_WARMUP = 500
NUM_SAMPLES = 500
CHUNK_SIZE = 10

model_std = phu.make_model(beta=1.0, tau0=tau0)
model_beta13 = phu.make_model(beta=1.3, tau0=tau0, k_trunc=250)

gen_poisson = functools.partial(phu.generate_poisson_data, n, p, b0_true, b_true)
gen_negbin = functools.partial(phu.generate_negbin_data, n, p, b0_true, b_true, R_NB)

/Users/alya57/venvs/myenv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Well-specified threshold (tau*)

Both datasets in each pair are drawn from the model's own assumed likelihood
(Poisson at the true b0,b) -- Q = point mass at the true parameters, Definition 1.
Standard Bayes (beta=1) posteriors are fit on each side via NUTS, and the
Bhattacharyya-coefficient discrepancy d_1/2 = -2 log BC(Pi1,Pi2) is estimated via a
two-directional bridge estimator, averaged over R replicate pairs.

In [2]:
pairs_wellspec = phu.generate_replicate_pairs(gen_poisson, R=R, base_seed=100)
result_tau_star = phu.pairwise_discrepancy_mc(
    model_std, pairs_wellspec, num_warmup=NUM_WARMUP, num_samples=NUM_SAMPLES,
    chunk_size=CHUNK_SIZE, base_seed_mcmc=1000, collect_meff=True, p_dim=p)

print(f"tau* = {result_tau_star['mean_d']:.4f} (SE={result_tau_star['se_d']:.4f}), "
      f"divergence rate={result_tau_star['div_rate']:.4f}")

/Users/alya57/venvs/myenv/lib/python3.13/site-packages/jax/_src/interpreters/mlir.py:1280: UserWarning: Some donated buffers were not usable: bool[500].
See an explanation at https://docs.jax.dev/en/latest/faq.html#buffer-donation.
  warnings.warn("Some donated buffers were not usable:"


  [10/30] mean_d=3.4030  std=1.2053  div_rate=0.0007  elapsed=2.4min  eta=4.7min


  [20/30] mean_d=3.3317  std=1.0159  div_rate=0.0016  elapsed=4.7min  eta=2.3min


  [30/30] mean_d=3.5957  std=1.2354  div_rate=0.0013  elapsed=7.0min  eta=0.0min
tau* = 3.5957 (SE=0.2256), divergence rate=0.0013


## Effective dimension check (Piironen-Vehtari m_eff)

The naive asymptotic reference tau*=D/2 (D=p+1=11, the number of structurally real
parameters (b0,b)) assumes a regular parameter vector; horseshoe's shrinkage means
the *effective* dimension is typically lower, since coefficients confidently
identified as null contribute less "effective complexity" than a free parameter
would. m_eff is computed directly from the same posterior draws used for tau* above
(Fisher-information analogue of Piironen & Vehtari's kappa_j, adapted from Gaussian
regression's n/sigma^2 to the Poisson-GLM's Sum_i mu_i x_ij^2).

In [3]:
print(f"m_eff = {result_tau_star['m_eff_mean']:.3f} +/- {result_tau_star['m_eff_se']:.3f}")
print(f"D_eff = m_eff + 1 (intercept) = {result_tau_star['D_eff']:.3f}")
print(f"tau* predicted from m_eff = D_eff/2 = {result_tau_star['tau_pred_from_meff']:.3f}")
print(f"tau* observed (bridge estimator, above) = {result_tau_star['mean_d']:.3f}")

m_eff = 6.394 +/- 0.049
D_eff = m_eff + 1 (intercept) = 7.394
tau* predicted from m_eff = D_eff/2 = 3.697
tau* observed (bridge estimator, above) = 3.596


## Observed discrepancy under standard Bayes (d_obs_std)

Same model (still assumes a Poisson likelihood), but now both datasets in each pair
are drawn from the TRUE, misspecified DGP: Negative-Binomial(r=5) with the same mean
mu_i as the well-specified case but genuine overdispersion (Var = mu + mu^2/r).

In [4]:
pairs_misspec = phu.generate_replicate_pairs(gen_negbin, R=R, base_seed=500)
result_d_obs_std = phu.pairwise_discrepancy_mc(
    model_std, pairs_misspec, num_warmup=NUM_WARMUP, num_samples=NUM_SAMPLES,
    chunk_size=CHUNK_SIZE, base_seed_mcmc=5000)

print(f"d_obs_std = {result_d_obs_std['mean_d']:.4f} (SE={result_d_obs_std['se_d']:.4f}), "
      f"divergence rate={result_d_obs_std['div_rate']:.4f}")

  [10/30] mean_d=25.9871  std=14.8322  div_rate=0.0003  elapsed=2.3min  eta=4.6min


  [20/30] mean_d=26.2598  std=14.5915  div_rate=0.0005  elapsed=4.5min  eta=2.3min


  [30/30] mean_d=26.8411  std=13.7676  div_rate=0.0005  elapsed=6.8min  eta=0.0min
d_obs_std = 26.8411 (SE=2.5136), divergence rate=0.0005


## Observed discrepancy under beta-Bayes (d_obs_beta, beta=1.3)

Same replicate pairs as d_obs_std above (paired comparison, no extra Monte Carlo
noise between the standard- and beta-Bayes discrepancies), but each posterior is now
fit under the beta-divergence loss (Basu et al.) instead of the exact likelihood:
ell_beta(y,theta) = -1/(beta-1) f(y;theta)^(beta-1) + 1/beta Sum_k f(k;theta)^beta.
Only the second (normalizer-like) term needs truncation (k_trunc=250, verified
numerically sufficient for this DGP's y/mu range); the first term is exact.

In [5]:
result_d_obs_beta13 = phu.pairwise_discrepancy_mc(
    model_beta13, pairs_misspec, num_warmup=NUM_WARMUP, num_samples=NUM_SAMPLES,
    chunk_size=CHUNK_SIZE, base_seed_mcmc=5000)

print(f"d_obs_beta(1.3) = {result_d_obs_beta13['mean_d']:.4f} (SE={result_d_obs_beta13['se_d']:.4f}), "
      f"divergence rate={result_d_obs_beta13['div_rate']:.4f}")

  [10/30] mean_d=3.0812  std=1.7417  div_rate=0.0000  elapsed=113.9min  eta=227.7min


  [20/30] mean_d=3.1162  std=1.3761  div_rate=0.0000  elapsed=159.4min  eta=79.7min


  [30/30] mean_d=3.1482  std=1.2337  div_rate=0.0000  elapsed=187.1min  eta=0.0min
d_obs_beta(1.3) = 3.1482 (SE=0.2252), divergence rate=0.0000


## Summary

In [6]:
print('=' * 70)
print(f"{'Measure':<30}{'mean d':>12}{'SE':>10}{'div rate':>12}")
print('-' * 70)
print(f"{'tau* (well-specified)':<30}{result_tau_star['mean_d']:>12.4f}{result_tau_star['se_d']:>10.4f}{result_tau_star['div_rate']:>12.4f}")
print(f"{'tau* predicted (m_eff)':<30}{result_tau_star['tau_pred_from_meff']:>12.4f}{'':>10}{'':>12}")
print(f"{'d_obs_std (beta=1)':<30}{result_d_obs_std['mean_d']:>12.4f}{result_d_obs_std['se_d']:>10.4f}{result_d_obs_std['div_rate']:>12.4f}")
print(f"{'d_obs_beta(1.3)':<30}{result_d_obs_beta13['mean_d']:>12.4f}{result_d_obs_beta13['se_d']:>10.4f}{result_d_obs_beta13['div_rate']:>12.4f}")
print('=' * 70)

Measure                             mean d        SE    div rate
----------------------------------------------------------------------
tau* (well-specified)               3.5957    0.2256      0.0013
tau* predicted (m_eff)              3.6970                      
d_obs_std (beta=1)                 26.8411    2.5136      0.0005
d_obs_beta(1.3)                     3.1482    0.2252      0.0000
